In [ ]:
%pip install -q dspy>=3.0.4 openpyxl
%pip install -q "/lakehouse/default/Files/fabric_rlm_longcot/wheels/fabric_rlm-0.2.1.dev2+excelskill-py3-none-any.whl"


In [ ]:
import os, sys, json, time, tarfile, shutil, subprocess, traceback, pathlib, re, urllib.request, hashlib
import openpyxl
import dspy
import fabric_rlm
from fabric_rlm import RLM, FabricLM

STRATEGY = "A"
MODEL    = "gpt-5"
EFFORT   = "medium"
RUN_ID   = "ssb-full400-A-20260505"
SMOKE_N  = 0

DATASET_TAR        = "/lakehouse/default/Files/fabric_rlm_longcot/datasets/ssb_full_400.tar.gz"
DATASET_JSONL_NAME = "ssb_full_400.jsonl"
DATASET_HF_URL     = "https://huggingface.co/datasets/KAKA22/SpreadsheetBench/resolve/main/spreadsheetbench_verified_400.tar.gz"
RUN_ROOT    = pathlib.Path("/lakehouse/default/Files/fabric_rlm_adaptive_validation/spreadsheetbench/ssb-full400-A-20260505")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RUN_ROOT / f"results_{STRATEGY}.jsonl"
SUMMARY_PATH = RUN_ROOT / f"summary_{STRATEGY}.json"
TRACES_DIR   = RUN_ROOT / f"traces_{STRATEGY}"
TRACES_DIR.mkdir(exist_ok=True)

def stage(event, **kw):
    print(json.dumps({"t": round(time.time(),2), "event": event, **kw}, default=str))

stage("imports_done", fabric_rlm=getattr(fabric_rlm, '__version__', '?'))

WORK = pathlib.Path("/tmp/ssb_work")
WORK.mkdir(parents=True, exist_ok=True)
DS_DIR = WORK / "ds"

# ---- Self-bootstrap dataset ----
# 1. If OneLake cache (DATASET_TAR + jsonl) is present, just extract.
# 2. Else, download the official SpreadsheetBench Verified-400 release from HuggingFace,
#    repackage the inner spreadsheets/ dir + a flat jsonl manifest, and cache to OneLake
#    so subsequent runs skip the download.
def _bootstrap_dataset():
    tar_p   = pathlib.Path(DATASET_TAR)
    jsonl_p = tar_p.parent / DATASET_JSONL_NAME
    if tar_p.exists() and jsonl_p.exists():
        stage("dataset_cache_hit", tar=str(tar_p))
        return tar_p, jsonl_p
    stage("dataset_cache_miss_downloading_hf", url=DATASET_HF_URL)
    tar_p.parent.mkdir(parents=True, exist_ok=True)
    tmp_raw = WORK / "ssb_hf_raw.tar.gz"
    if not tmp_raw.exists():
        urllib.request.urlretrieve(DATASET_HF_URL, tmp_raw)
    raw_dir = WORK / "ssb_hf_raw"
    raw_dir.mkdir(exist_ok=True)
    with tarfile.open(tmp_raw) as tf:
        tf.extractall(raw_dir)
    inner = next(raw_dir.glob("spreadsheetbench_verified_400*"), None) or raw_dir
    if not (inner / "dataset.json").exists():
        cands = list(raw_dir.rglob("dataset.json"))
        if cands:
            inner = cands[0].parent
    ds_json = json.load(open(inner / "dataset.json", encoding="utf-8"))
    spr_dir = inner / "spreadsheet"
    # Build flat manifest: pick first init/golden pair per record.
    records = []
    for rec in ds_json:
        sid = str(rec["id"])
        d = spr_dir / sid
        if not d.exists():
            continue
        inits   = sorted(d.glob("*_init.xlsx"))
        goldens = sorted(d.glob("*_golden.xlsx"))
        if not (inits and goldens):
            continue
        prompt_p = d / "prompt.txt"
        instr = prompt_p.read_text(encoding="utf-8") if prompt_p.exists() else rec.get("instruction","")
        records.append({
            "question_id": f"SSB_{sid}",
            "spreadsheet_id": sid,
            "instruction": instr,
            "instruction_type": rec.get("instruction_type"),
            "answer_position": rec["answer_position"],
            "answer_sheet":    rec.get("answer_sheet"),
            "init_file":   inits[0].name,
            "golden_file": goldens[0].name,
        })
    with open(jsonl_p, "w", encoding="utf-8") as fh:
        for r in records:
            fh.write(json.dumps(r) + "\n")
    # Repack flattened bundle: jsonl will be re-emitted; tarball contains only spreadsheets/.
    stage_dir = WORK / "ssb_stage"
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    stage_dir.mkdir()
    shutil.copytree(spr_dir, stage_dir / "spreadsheets")
    with tarfile.open(tar_p, "w:gz") as tf:
        tf.add(stage_dir / "spreadsheets", arcname="spreadsheets")
    stage("dataset_bootstrapped", n_records=len(records),
          tar_bytes=tar_p.stat().st_size, jsonl=str(jsonl_p))
    return tar_p, jsonl_p

DATASET_TAR_P, DATASET_JSONL_P = _bootstrap_dataset()

if not DS_DIR.exists():
    DS_DIR.mkdir()
    with tarfile.open(DATASET_TAR_P) as tf:
        tf.extractall(DS_DIR)
n_xlsx = len(list(DS_DIR.rglob('*.xlsx')))
stage("dataset_extracted", path=str(DS_DIR), xlsx_count=n_xlsx)

records = [json.loads(l) for l in open(DATASET_JSONL_P, encoding='utf-8')]
if SMOKE_N > 0:
    records = records[:SMOKE_N]
stage("records_loaded", n=len(records))

def grade(out_xlsx, gold_xlsx, sheet, cell_range):
    try:
        wb_a = openpyxl.load_workbook(out_xlsx, data_only=True)
        wb_b = openpyxl.load_workbook(gold_xlsx, data_only=True)
    except Exception as e:
        return False, 0, 0, f"load_err: {e}"
    sname_a = sheet if (sheet and sheet in wb_a.sheetnames) else wb_a.sheetnames[0]
    sname_b = sheet if (sheet and sheet in wb_b.sheetnames) else wb_b.sheetnames[0]
    a = wb_a[sname_a]; b = wb_b[sname_b]
    try:
        rng_a = a[cell_range]; rng_b = b[cell_range]
    except Exception as e:
        return False, 0, 0, f"range_err: {e}"
    flat_a = [c.value for row in rng_a for c in row]
    flat_b = [c.value for row in rng_b for c in row]
    if len(flat_a) != len(flat_b):
        return False, 0, len(flat_b), f"len_mismatch {len(flat_a)} vs {len(flat_b)}"
    matches = 0
    for x, y in zip(flat_a, flat_b):
        if (x is None and y is None) or (str(x).strip() == str(y).strip()):
            matches += 1
    return matches == len(flat_a), matches, len(flat_a), None


In [ ]:
os.environ["FABRIC_RLM_CAPTURE_TURNS"] = "1"
base_lm = FabricLM("gpt-5", reasoning_effort="medium", max_tokens=16000)
stage("lm_built", model="gpt-5", effort="medium")


In [ ]:
dspy.configure(lm=base_lm)

class SsbWriteCode(dspy.Signature):
    """Given an Excel manipulation instruction and the path to a working .xlsx,
    write a complete Python program that uses openpyxl (and pandas/numpy if helpful)
    to perform the manipulation IN-PLACE and save the workbook back to the same path.

    CRITICAL: The grader reads cell values with openpyxl(data_only=True), which does
    NOT evaluate Excel formulas. You MUST write the COMPUTED NUMERIC/STRING VALUES
    into the target cells, not Excel formulas. Compute everything in Python and write
    the literal result.
    """
    instruction: str = dspy.InputField()
    xlsx_path: str   = dspy.InputField()
    sheet_hint: str  = dspy.InputField(desc="Target sheet name; may be empty")
    answer_range: str = dspy.InputField(desc="Cell range that the grader will inspect, e.g. A3:D32")
    code: str = dspy.OutputField(desc="ONLY a Python program inside a ```python``` fenced block. Must end by saving the workbook back to xlsx_path. Write computed values, NOT Excel formulas.")

predict = dspy.Predict(SsbWriteCode)

def extract_code(text):
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text or "", re.DOTALL)
    return (m.group(1) if m else (text or "")).strip()

t_start = time.time()
n_pass = 0
with RESULTS_PATH.open("w", encoding="utf-8") as out_fh:
    for idx, rec in enumerate(records):
        qid = rec['question_id']; sid = str(rec['spreadsheet_id'])
        init_src = DS_DIR / 'spreadsheets' / sid / rec['init_file']
        gold_src = DS_DIR / 'spreadsheets' / sid / rec['golden_file']
        work_dir = WORK / qid; work_dir.mkdir(exist_ok=True)
        work_xlsx = work_dir / 'work.xlsx'
        result = {"strategy": STRATEGY, "question_id": qid, "spreadsheet_id": sid,
                  "instruction_type": rec['instruction_type'],
                  "answer_sheet": rec.get('answer_sheet'),
                  "answer_position": rec['answer_position']}
        try:
            shutil.copyfile(init_src, work_xlsx)
            t0 = time.perf_counter()
            pred = predict(instruction=rec['instruction'], xlsx_path=str(work_xlsx),
                           sheet_hint=rec.get('answer_sheet') or '',
                           answer_range=rec['answer_position'])
            elapsed_lm = time.perf_counter() - t0
            code = extract_code(pred.code)
            history = getattr(base_lm, "history", []) or []
            last = history[-1] if history else {}
            usage = (last.get("usage") or {}) if isinstance(last, dict) else {}
            prompt_tok = usage.get("prompt_tokens") or usage.get("input_tokens") or 0
            completion_tok = usage.get("completion_tokens") or usage.get("output_tokens") or 0
            # Run code in subprocess
            t1 = time.perf_counter()
            p = subprocess.run([sys.executable, "-c", code],
                               capture_output=True, text=True, timeout=180)
            elapsed_exec = time.perf_counter() - t1
            (TRACES_DIR / f"trace_{qid}.json").write_text(json.dumps({
                "qid": qid, "instruction": rec['instruction'],
                "code": code, "returncode": p.returncode,
                "stdout": p.stdout[-2000:], "stderr": p.stderr[-2000:],
            }, indent=2), encoding='utf-8')
            passed, m, n, gerr = grade(str(work_xlsx), str(gold_src),
                                       rec.get('answer_sheet') or '', rec['answer_position'])
            result.update({
                "passed": passed, "cells_matched": m, "cells_total": n,
                "grade_err": gerr,
                "lm_seconds": round(elapsed_lm,2), "exec_seconds": round(elapsed_exec,2),
                "subprocess_returncode": p.returncode,
                "stderr_tail": p.stderr[-300:],
                "prompt_tokens": prompt_tok, "completion_tokens": completion_tok,
            })
        except Exception as e:
            result.update({"passed": False, "error": repr(e),
                           "traceback": traceback.format_exc()[:500]})
        if result.get("passed"): n_pass += 1
        out_fh.write(json.dumps(result, default=str) + "\n"); out_fh.flush()
        stage("q_done", idx=idx+1, qid=qid, passed=result.get("passed"),
              cells=f"{result.get('cells_matched',0)}/{result.get('cells_total',0)}",
              lm_s=result.get("lm_seconds"), exec_s=result.get("exec_seconds"))

summary = {"strategy": STRATEGY, "model": MODEL, "effort": EFFORT, "run_id": RUN_ID,
           "n": len(records), "n_passed": n_pass,
           "pass_rate": round(n_pass/max(1,len(records)), 4),
           "total_seconds": round(time.time()-t_start, 1)}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))
